# LongMemEval domain fine-tune (T4 Colab)

Self-contained: clones the experiment branch, downloads LongMemEval-s
from HuggingFace (~265 MB), runs the data prep + ingest, then trains
with `vstash retrain --training-queries ... --eval-queries ...`
(non-synthetic, labeled v5 recipe). No Drive upload needed.

Reported as a SEPARATE result from the LongMemEval baseline
retrieval numbers: this is a domain-adaptation demonstration
("training on chat data improves chat retrieval"), not a way to
inflate v3's headline number.

Pipeline:
  1. Setup: clone the `feature/longmemeval-retrain-experiments`
     branch and `pip install -e .`.
  2. Download longmemeval_s from `xiaowu0162/longmemeval` on HF.
  3. Run `experiments.lme_prepare_retrain` to build the corpus DB +
     stratified train/holdout JSONLs.
  4. Run `vstash retrain` with `--training-queries` and the new
     `--eval-queries` flag (PR #299, merged to develop).
  5. Save the candidate to Drive so the local Mac can pull it and
     score `experiments/longmemeval_retrieval` against the same 500
     questions used for the v3 baseline.

In [ ]:
# Cell 1: Setup -- clone the branch with the LongMemEval experiment
# scripts (lme_prepare_retrain, longmemeval_retrieval).  --eval-queries
# is on develop already (PR #299) so any develop-or-later branch works.
BRANCH = 'feature/longmemeval-retrain-experiments'

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' huggingface_hub
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .
!vstash --version
!vstash retrain --help | grep -E '\-\-training-queries|\-\-eval-queries' | head -4

In [ ]:
# Cell 2: Download longmemeval_s (~265 MB) from HF directly into the
# experiments/data/longmemeval/ path that the prep script expects.
import os
from huggingface_hub import hf_hub_download

TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)

path = hf_hub_download(
    repo_id='xiaowu0162/longmemeval',
    filename='longmemeval_s',
    repo_type='dataset',
    local_dir=TARGET,
)
print(f'Downloaded longmemeval_s ({os.path.getsize(path) / 1024 / 1024:.1f} MB) -> {path}')

In [ ]:
# Cell 3: Build the corpus DB + train/eval JSONLs.
# Stratified 80/20 split by question_type, deterministic seed=42.
# ~9 min on Colab CPU before the GPU phase begins (ingest is
# embedding-bound; the GPU step is bulk_mine + train).
import os
os.chdir('/content/vstash')

!python -m experiments.lme_prepare_retrain \
    --output-db    /content/lme_corpus.db \
    --output-train /content/lme_train.jsonl \
    --output-eval  /content/lme_eval.jsonl \
    --output-meta  /content/lme_retrain_meta.json \
    --force

import json
meta = json.load(open('/content/lme_retrain_meta.json'))
print()
print('Split:')
for k in ('n_train_questions', 'n_holdout_questions', 'n_train_qrels',
         'n_eval_qrels', 'n_train_docs', 'n_holdout_docs'):
    print(f'  {k:25s} {meta[k]}')
print('By type:')
for t, counts in meta['by_type'].items():
    print(f'  {t:30s} train={counts["train"]:3d}  holdout={counts["holdout"]:3d}')

In [ ]:
# Cell 4: GPU + retrain config.
import os, torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0),
          '| mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

BASE_MODEL = 'BAAI/bge-small-en-v1.5'
OUTPUT_PATH = '/content/bge-small-rrf-lme-v1'
MAX_QUERIES = 5000
EPOCHS = 2
LR = 3e-6
BATCH = 64
SEED = 42

os.environ['VSTASH_DB_PATH'] = '/content/lme_corpus.db'
print('VSTASH_DB_PATH ->', os.environ['VSTASH_DB_PATH'])

In [ ]:
# Cell 5: Run vstash retrain.  Pipeline is:
#   - generate_labeled_triples_batched on T4,
#   - train MNRL with hard negatives,
#   - evaluate_model baseline + final NDCG@10 on the 102 holdout
#     queries (--eval-queries flag, PR #299).
# Gate refuses to save if NDCG@10 does not improve (--min-gain 0.0).
import time
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_MODEL \
    --output           $OUTPUT_PATH \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain base BGE] wall: {time.perf_counter() - t0:.1f}s')

In [ ]:
# Cell 6: Persist the trained model to Drive so the local Mac can
# download it and run the full retrieval-side benchmark
# (experiments.longmemeval_retrieval --model <path>) against the
# same 500 questions used for the v3 baseline.
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_retrain'
os.makedirs(DRIVE_OUT, exist_ok=True)

for suffix in ('', '.candidate', '.old'):
    src = OUTPUT_PATH + suffix
    if os.path.isdir(src):
        dst = os.path.join(DRIVE_OUT, f'bge-small-rrf-lme-v1{suffix}')
        !cp -r "$src" "$dst"
        print(f'Copied {src} -> {dst}')
    else:
        print(f'(skip) {src} does not exist')
!ls -la $DRIVE_OUT | grep lme-v1

In [ ]:
# Cell 7 (optional): try v3 as base model.  v3 is published as a
# SentenceTransformer-loadable repo on HF, so we can stack the
# domain fine-tune on top of it instead of starting from base BGE.
# The two arms answer different questions:
#   base BGE -> bge-small-rrf-lme-v1: pure 'chat data lifts BGE'.
#   v3       -> bge-small-rrf-lme-v1-from-v3: 'chat data on top of
#               BEIR-tuned weights still lifts further?'.
import time
OUTPUT_V3 = '/content/bge-small-rrf-lme-v1-from-v3'
BASE_V3   = 'Stffens/bge-small-rrf-v3'
t0 = time.perf_counter()
!vstash retrain \
    --training-queries /content/lme_train.jsonl \
    --eval-queries     /content/lme_eval.jsonl \
    --base-model       $BASE_V3 \
    --output           $OUTPUT_V3 \
    --max-queries      $MAX_QUERIES \
    --epochs           $EPOCHS \
    --lr               $LR \
    --batch-size       $BATCH \
    --seed             $SEED \
    --bulk-mine        \
    --bulk-mine-device cuda
print(f'\n[retrain v3-base] wall: {time.perf_counter() - t0:.1f}s')

import os
for suffix in ('', '.candidate'):
    src = OUTPUT_V3 + suffix
    if os.path.isdir(src):
        dst = os.path.join(DRIVE_OUT, f'bge-small-rrf-lme-v1-from-v3{suffix}')
        !cp -r "$src" "$dst"
        print(f'Copied {src} -> {dst}')